# BattleActs Actuarial Exam Preparation Products
### DATASCI 530 — Project 1
Maxine Hakimi-Bulson

This project compares six BattleActs exam preparation courses: Exams 5, 6C, 6U, 7, 8, and 9. It collects listed prices, access periods, practice-exam counts, advertised study tools, and selected public sample contents.

Data were collected on October 19, 2026. BattleActs granted written permission for this educational scraping project and for sharing the code, dataset, and analysis on GitHub. Collection was limited to public product pages and selected public PDF and Excel samples. Accounts, paid course materials, customer data, and video downloads were excluded. Requests were spaced with pauses.


## Data Collection Process

Before coding, I manually inspected two products and organized their information in a spreadsheet to choose the variables.

The script opens the Exams menu, identifies six product links, and uses a loop to click each link and extract its product information and sample cards. It then downloads the selected public manual PDFs and Excel samples once per unique URL, reusing existing local files on later runs. PDF text and spreadsheet cell values are extracted; spreadsheet formulas are read as text and are not executed.

Two datasets are exported: Products contains one row per exam course; Samples contains one row per advertised sample card, including extracted text where selected. The product URL links them. Tool indicators record mentions in collected product descriptions, feature lists, and sample labels. “Not matched” does not establish that a feature is unavailable. Sample formats are inferred from URL extensions; only selected files were downloaded and parsed.


In [1]:
%pip install requests pypdf openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import packages
import pandas as pd
import time
from selenium import webdriver
import re
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import requests
from pathlib import Path
from urllib.parse import urlsplit, unquote
from pypdf import PdfReader
from openpyxl import load_workbook

# Open Chrome and navigate to BattleActs
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

starting_url = "https://battleactsmain.ca/v2/home"
driver.get(starting_url)

## Product Extraction Function

In [3]:
def extract_product(driver):
    # Return None when an optional text field is absent
    def get_text(xpath):
        elements = driver.find_elements("xpath", xpath)
        return elements[0].text.strip() if elements else None

    # Basic product information
    product_name = get_text("//h1")
    price_text = get_text('//h2[contains(., "$")]')

    price = (
        float(price_text.split()[0].replace("$", "").replace(",", ""))
        if price_text else None
    )
    currency = price_text.split()[-1] if price_text else None

    # Description and its bullet points
    description_parts = driver.find_elements(
        "xpath",
        '//h2[contains(., "Study Guide")]/parent::div/p'
        ' | //h2[contains(., "Study Guide")]/parent::div//li'
    )
    description = "\n".join(
        element.text.strip() for element in description_parts
    ) or None

    # Overview feature bullets
    feature_items = driver.find_elements(
        "xpath",
        '//h2[normalize-space()="Overview Of Our Features"]'
        '/parent::div/parent::div//ul/li'
    )
    features = "\n".join(
        item.text.strip() for item in feature_items
    ) or None

    # Structured fields derived from the collected text
    feature_text = features or ""
    duration_match = re.search(
        r"(\d+)-month'?s?\s+access", feature_text, re.I
    )
    extension_match = re.search(
        r"free\s+(\d+)-month\s+extension", feature_text, re.I
    )
    practice_match = re.search(
        r"(\d+)\s+original practice exams?", feature_text, re.I
    )

    # Exam-specific preparation description
    exam_differences = get_text(
        '//p[contains(., "different than other")]'
        '/following-sibling::p[1]'
    )

    product_url = driver.current_url
    collection_date = pd.Timestamp.now(
        tz="America/New_York"
    ).isoformat(timespec="seconds")

    product = {
        "product_name": product_name,
        "price": price,
        "currency": currency,
        "description": description,
        "features": features,
        "duration_months": (
            int(duration_match.group(1)) if duration_match else None
        ),
        "product_url": product_url,
        "collection_date": collection_date,
        "exam_differences": exam_differences,
        "practice_exam_count": (
            int(practice_match.group(1)) if practice_match else None
        ),
        "free_extension_months": (
            int(extension_match.group(1)) if extension_match else None
        )
    }

    # Sample cards within the section containing the sample heading
    sample_cards = driver.find_elements(
        "xpath",
        '//h2[normalize-space()="Sample our Study Platform"]'
        '/ancestor::div[contains(concat(" ", normalize-space(@class), " "),'
        ' " product-section ")][1]'
        '//div[contains(concat(" ", normalize-space(@class), " "),'
        ' " owl-item ")'
        ' and not(contains(concat(" ", normalize-space(@class), " "),'
        ' " cloned "))]'
    )

    samples = []

    for card in sample_cards:
        labels = card.find_elements("xpath", ".//p")
        controls = card.find_elements(
            "xpath", ".//a[@href] | .//button[@data-url]"
        )

        if not labels or not controls:
            raise ValueError(
                "Unexpected sample-card structure on " + product_url
            )

        control = controls[0]
        sample_url = (
            control.get_attribute("href")
            or control.get_attribute("data-url")
        )
        url_path = (sample_url or "").split("?")[0].split("#")[0].lower()

        if url_path.endswith(".pdf"):
            resource_type = "PDF"
        elif url_path.endswith((".xlsx", ".xls")):
            resource_type = "Excel"
        elif url_path.endswith(".mp4"):
            resource_type = "Video"
        else:
            resource_type = "Unknown"

        samples.append({
            "product_name": product_name,
            "sample_label": labels[0].get_attribute("textContent").strip(),
            "button_text": control.get_attribute("textContent").strip(),
            "sample_url": sample_url,
            "resource_type": resource_type,
            "product_url": product_url,
            "collection_date": collection_date
        })

    return product, samples

## Automated Navigation and Collection

In [4]:
wait = WebDriverWait(driver, 15)

# Open the homepage and collect the exam-link destinations
driver.get(starting_url)

menu_xpath = '//a[@id="navbarDropdown" and normalize-space()="Exams"]'
links_xpath = (
    '//a[contains(@class, "dropdown-item")'
    ' and contains(@href, "/v2/product/")]'
)

wait.until(EC.element_to_be_clickable((By.XPATH, menu_xpath))).click()

exam_links = driver.find_elements("xpath", links_xpath)

# Remove repeated URLs while preserving their order
exam_urls = list(dict.fromkeys(
    link.get_attribute("href") for link in exam_links
))

if len(exam_urls) != 6:
    raise ValueError(
        f"Expected 6 exam links, but found {len(exam_urls)}. Check the menu."
    )

products = []
all_samples = []

for exam_url in exam_urls:
    # Reopen the homepage and menu for each product
    driver.get(starting_url)
    wait.until(EC.element_to_be_clickable((By.XPATH, menu_xpath))).click()

    # Navigate by actually clicking the product link
    product_link_xpath = (
        '//a[contains(@class, "dropdown-item")'
        f' and @href="{exam_url}"]'
    )
    wait.until(
        EC.element_to_be_clickable((By.XPATH, product_link_xpath))
    ).click()

    wait.until(EC.url_to_be(exam_url))
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h1")))
    time.sleep(2)  # Allow the sample carousel to initialize

    product, samples = extract_product(driver)

    # Preserve completed rows as the loop progresses
    products.append(product)
    all_samples.extend(samples)

    print(
        f'{product["product_name"]}: '
        f'{product["price"]} {product["currency"]}; '
        f'{len(samples)} sample cards'
    )

    time.sleep(3)  # Pause between products

# Build the two datasets
products_df = pd.DataFrame(products)
samples_df = pd.DataFrame(all_samples)

print("Total sample cards:", len(samples_df))

CAS Exam 5: 500.0 USD; 6 sample cards
CAS Exam 6C: 1000.0 CAD; 3 sample cards
CAS Exam 6U: 1000.0 USD; 6 sample cards
CAS Exam 7: 1000.0 USD; 4 sample cards
CAS Exam 8: 1000.0 USD; 3 sample cards
CAS Exam 9: 1000.0 USD; 3 sample cards
Total sample cards: 25


## Prepare Study-Tool Categories

In [5]:
has_url = samples_df["sample_url"].fillna("").str.strip().ne("")
coming_soon = samples_df["button_text"].str.contains("coming soon", case=False, na=False)
samples_df["sample_status"] = "No URL listed"
samples_df.loc[has_url, "sample_status"] = "Link listed"
samples_df.loc[~has_url & coming_soon, "sample_status"] = "Coming soon"
# Classify selected tools using product text and sample labels
sample_labels = samples_df.groupby("product_url")["sample_label"].agg("\n".join)
tool_text = (
    products_df[["description", "features"]].fillna("").agg("\n".join, axis=1)
    + "\n" + products_df["product_url"].map(sample_labels).fillna("")
)
products_final = products_df.copy()
for column, pattern in {
    "instructional_video_mentioned": r"video lessons|instructional videos?",
    "formula_summary_sheet_mentioned": r"formula|summary sheets?",
    "hybrid_exam_mentioned": r"hybrid CAS exam"
}.items():
    products_final[column] = tool_text.str.contains(pattern, case=False, na=False)


## Extract Public Sample Contents

In [6]:
# Select public manual PDFs and linked Excel samples
manual_pdf = (
    samples_df["resource_type"].eq("PDF")
    & samples_df["sample_label"].str.contains("study manual", case=False, na=False)
)
selected_samples = samples_df.loc[
    (manual_pdf | samples_df["resource_type"].eq("Excel"))
    & samples_df["sample_url"].fillna("").str.strip().ne("")
].copy()

sample_folder = Path("sample_files")
sample_folder.mkdir(exist_ok=True)
content_records = []

for number, url in enumerate(selected_samples["sample_url"].drop_duplicates(), start=1):
    suffix = Path(unquote(urlsplit(url).path)).suffix.lower()
    local_path = sample_folder / f"sample_{number:02d}{suffix}"
    record = {
        "sample_url": url,
        "content_collection_date": pd.Timestamp.now(tz="America/New_York").isoformat(timespec="seconds"),
        "extraction_status": "Failed",
        "page_count": None, "sheet_count": None,
        "sheet_names": None, "formula_cells": None, "extracted_text": None
    }

    try:
        if not local_path.exists():
            response = requests.get(
                url, headers={"User-Agent": "DATASCI530 educational project"}, timeout=60
            )
            response.raise_for_status()
            local_path.write_bytes(response.content)

        if suffix == ".pdf":
            reader = PdfReader(local_path)
            record["page_count"] = len(reader.pages)
            text = "\n".join(page.extract_text() or "" for page in reader.pages)

        elif suffix == ".xlsx":
            workbook = load_workbook(local_path, read_only=True, data_only=False)
            try:
                record["sheet_count"] = len(workbook.sheetnames)
                record["sheet_names"] = " | ".join(workbook.sheetnames)
                text_parts = []
                formula_count = 0
                for sheet in workbook.worksheets:
                    text_parts.append(f"Worksheet: {sheet.title}")
                    for row in sheet.iter_rows():
                        values = []
                        for cell in row:
                            if cell.value is not None:
                                values.append(str(cell.value))
                                formula_count += int(cell.data_type == "f")
                        if values:
                            text_parts.append(" | ".join(values))
                record["formula_cells"] = formula_count
                text = "\n".join(text_parts)
            finally:
                workbook.close()
        else:
            raise ValueError(f"Unsupported format: {suffix}")

        record["extracted_text"] = text.strip() or None
        record["extraction_status"] = "Extracted" if text.strip() else "No text extracted"

    except Exception as error:
        print(f"Could not extract {local_path.name}: {type(error).__name__}: {error}")

    content_records.append(record)
    print(local_path.name, record["extraction_status"])
    time.sleep(3)

content_df = pd.DataFrame(content_records)


sample_01.pdf Extracted
sample_02.xlsx Extracted
sample_03.pdf Extracted
sample_04.xlsx Extracted
sample_05.pdf Extracted
sample_06.xlsx Extracted
sample_07.xlsx Extracted
sample_08.pdf Extracted
sample_09.xlsx Extracted
sample_10.pdf Extracted
sample_11.xlsx Extracted
sample_12.pdf Extracted


## Build and Check Final Datasets

In [7]:
# Attach extracted contents without losing unselected or Coming Soon cards
samples_final = samples_df.drop(columns="button_text").merge(
    content_df, on="sample_url", how="left", validate="many_to_one"
)
not_selected = samples_final["extraction_status"].isna()
samples_final.loc[not_selected, "extraction_status"] = "Not selected"
samples_final.loc[
    not_selected & samples_final["sample_status"].eq("Coming soon"), "extraction_status"
] = "No URL listed"

assert products_final["product_url"].is_unique
assert len(products_final) == len(products_df)
assert len(samples_final) == len(samples_df)
assert not samples_final.duplicated().any()
assert samples_final["product_url"].isin(products_final["product_url"]).all()
assert samples_final["extraction_status"].eq("Extracted").sum() == content_df["extraction_status"].eq("Extracted").sum()


## Results

### Table 1. Exam Preparation Course Overview

The six courses below are the products collected in this project. Prices are listed in their original currencies. The extension is conditional; seven months of standard access should not be interpreted as thirteen months of guaranteed access.

In [8]:
# Course-by-course overview: each row is one exam preparation product
course_overview = products_final[[
    "product_name", "price", "currency", "duration_months",
    "practice_exam_count", "free_extension_months"
]].rename(columns={
    "product_name": "Exam prep course",
    "price": "Listed price",
    "currency": "Currency",
    "duration_months": "Access (months)",
    "practice_exam_count": "Original practice exams",
    "free_extension_months": "Conditional extension (months)"
})
course_overview["Listed price"] = course_overview["Listed price"].map("{:,.2f}".format)
display(course_overview)


,Exam prep course,Listed price,Currency,Access (months),Original practice exams,Conditional extension (months)
0,CAS Exam 5,500.00,USD,7,4,6
1,CAS Exam 6C,"1,000.00",CAD,7,1,6
2,CAS Exam 6U,"1,000.00",USD,7,2,6
3,CAS Exam 7,"1,000.00",USD,7,2,6
4,CAS Exam 8,"1,000.00",USD,7,2,6
5,CAS Exam 9,"1,000.00",USD,7,2,6


### Table 2. Dataset Completeness

Missing values include the Coming Soon URL, content not selected for extraction, and measurements that do not apply to a file format.

In [9]:
quality_records = []
for name, frame in {"Products": products_final, "Samples": samples_final}.items():
    blank_cells = frame.apply(
        lambda column: column.map(lambda value: isinstance(value, str) and not value.strip())
    )
    quality_records.append({
        "Dataset": name, "Observations": len(frame),
        "Missing/blank cells": int((frame.isna() | blank_cells).sum().sum())
    })
display(pd.DataFrame(quality_records))
print("Sample cards missing a URL:", samples_final["sample_url"].isna().sum())
print("Extraction outcomes:", samples_final["extraction_status"].value_counts().to_dict())


,Dataset,Observations,Missing/blank cells
0,Products,6,0
1,Samples,25,103


Sample cards missing a URL: 1
Extraction outcomes: {'Extracted': 12, 'Not selected': 12, 'No URL listed': 1}


## Data Quality

The collection contains six products and 25 sample cards, with no duplicate product URLs or sample rows. Original product fields are complete. One Exam 9 card has no URL because it is labeled “Coming Soon”; it is retained. Shared URLs are retained when different cards advertise them.

All 12 selected samples yielded text: six PDFs and six Excel workbooks. The remaining 12 linked cards were not selected for content extraction. Blank content fields for these cards and the Coming Soon card are expected. PDF and Excel measurements are format-specific: PDF page counts do not apply to spreadsheets, and worksheet and formula-cell counts do not apply to PDFs.

Exam 5's price, access period, extension, and practice-exam count were checked against the manual collection. Prices are numeric and currencies remain separate. Worksheet counts include instructions and contents sheets, so they are not question counts. Extracted text may omit images and visual layout.


### Table 3. Numeric Summaries

Variances are sample variances. Price variances use squared currency units. The CAD variance is undefined with one observation.

In [10]:
# Prices are kept separate by currency; other measures use all products
summary_records = []
for currency, group in products_final.groupby("currency"):
    values = group["price"]
    summary_records.append({"Measure": f"Price ({currency})",
                            "Observations": values.count(),
                            "Mean": values.mean(), "Sample variance": values.var()})
for column, label in {
    "duration_months": "Standard access (months)",
    "practice_exam_count": "Original practice exams",
    "free_extension_months": "Conditional extension (months)"
}.items():
    values = products_final[column]
    summary_records.append({"Measure": label, "Observations": values.count(),
                            "Mean": values.mean(), "Sample variance": values.var()})
display(pd.DataFrame(summary_records).round(2))


,Measure,Observations,Mean,Sample variance
0,Price (CAD),1,1000.00,NaN
1,Price (USD),5,900.00,50000.00
2,Standard access (months),6,7.00,0.00
3,Original practice exams,6,2.17,0.97
4,Conditional extension (months),6,6.00,0.00


### Table 4. Selected Study Tools

These indicators capture mentions, not verified availability. Features common to all courses remain in the original description text rather than repeated indicator columns.

In [11]:
tool_table = products_final.set_index("product_name")[[
    "instructional_video_mentioned", "formula_summary_sheet_mentioned", "hybrid_exam_mentioned"
]].rename(columns={
    "instructional_video_mentioned": "Instructional videos",
    "formula_summary_sheet_mentioned": "Formula/summary sheets",
    "hybrid_exam_mentioned": "Hybrid exam"
}).replace({True: "Mentioned", False: "Not matched"})
tool_table.index.name = "Exam prep course"
display(tool_table)


,Instructional videos,Formula/summary sheets,Hybrid exam
Exam prep course,,,
CAS Exam 5,Mentioned,Mentioned,Not matched
CAS Exam 6C,Not matched,Not matched,Not matched
CAS Exam 6U,Mentioned,Mentioned,Not matched
CAS Exam 7,Not matched,Mentioned,Mentioned
CAS Exam 8,Not matched,Not matched,Mentioned
CAS Exam 9,Not matched,Not matched,Mentioned


### Table 5. Selected Sample Characteristics

Each row is a selected sample. Blank format-specific measurements mean not applicable. Counts describe document structure, not course quality.

In [12]:
sample_overview = samples_final.loc[
    samples_final["extraction_status"].isin(["Extracted", "Failed", "No text extracted"]),
    ["product_name", "resource_type", "extraction_status", "page_count", "sheet_count", "formula_cells"]
].rename(columns={
    "product_name": "Exam prep course", "resource_type": "Format",
    "extraction_status": "Extraction", "page_count": "PDF pages",
    "sheet_count": "Worksheets", "formula_cells": "Formula cells"
})
for column in ["PDF pages", "Worksheets", "Formula cells"]:
    sample_overview[column] = sample_overview[column].astype("Int64")
display(sample_overview)


,Exam prep course,Format,Extraction,PDF pages,Worksheets,Formula cells
0,CAS Exam 5,PDF,Extracted,17,<NA>,<NA>
4,CAS Exam 5,Excel,Extracted,<NA>,32,325
6,CAS Exam 6C,PDF,Extracted,7,<NA>,<NA>
8,CAS Exam 6C,Excel,Extracted,<NA>,11,288
9,CAS Exam 6U,PDF,Extracted,5,<NA>,<NA>
11,CAS Exam 6U,Excel,Extracted,<NA>,29,275
14,CAS Exam 6U,Excel,Extracted,<NA>,3,70
15,CAS Exam 7,PDF,Extracted,9,<NA>,<NA>
18,CAS Exam 7,Excel,Extracted,<NA>,4,42
19,CAS Exam 8,PDF,Extracted,6,<NA>,<NA>


## Discussion

Five courses are priced in USD, averaging $900; Exam 6C is listed separately at CAD 1,000. All six list seven months of standard access and a conditional six-month extension. Original practice-exam counts range from one to four.

The collected text shows differences in mentions of instructional videos, formula/summary sheets, and hybrid exams. The 25 public sample cards include 24 listed destinations and one Coming Soon card. Text was extracted from 12 selected documents for later content comparisons.

These data describe one provider at one point in time. Advertised features and selected excerpts do not measure course quality, exam difficulty, or student outcomes. Sample sizes and document structures are not directly comparable measures of course coverage.


## Export Final Datasets

In [13]:
products_final.to_csv("battleacts_products.csv", index=False)
samples_final.to_csv("battleacts_samples.csv", index=False)

In [14]:
driver.quit()